# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oumaklaus/ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (one of the four predefined core lanes).

I picked this lane for three reasons. First, it maps onto a decision a real person makes every week: an editor with limited time has to choose *which pages to review and refresh first*. Second, the starter dataset ships exactly the signals this lane needs — 90-day search and engagement metrics, freshness (`days_since_last_update`), and a decline signal — so I can back the choice with evidence today, not hand-wave. Third, the reference pipeline in `scripts/` is already built around this lane, so as the weeks go on (signal audit -> baseline -> model -> validation -> action playbook) I always have an honest reference to compare my own work against, instead of inventing scaffolding from scratch. The output is concrete: a **ranked review queue** with a priority score and human-readable **reason codes** per page.

In [1]:
# Load the small anonymized starter dataset that ships in this repo.
# One row = one pseudonymized content item (page); metrics are trailing-90-day.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"rows (pages):       {len(df):,}")
print(f"columns:            {df.shape[1]}")
print(f"clients (grouping): {df['client_id'].nunique()}")
# Enough scale and enough clients to do grouped (client-holdout) validation later.

rows (pages):       30,000
columns:            44
clients (grouping): 32


## 2. The question: decision, action, cost of a wrong call

**The search question.** Given a large library of published pages and an editor who can only review a handful each week, *which pages should be reviewed and refreshed first?*

Framed against the four questions from `framing-ml-problems`:

- **Decision it improves.** How an editor spends limited review time — which pages get looked at this week rather than next quarter (or never).
- **Unit of analysis (grain).** One **page** (one `content_id`), as it stands in the trailing-90-day window. Not a client, not a day.
- **Output.** A ranked review queue: a priority *score* per page plus short **reason codes** (e.g. `declining_with_demand`, `low_ctr_visible_page`) that tell the reviewer *why* a page surfaced.
- **Who acts, and how.** A content editor / SEO reviewer opens the top of the queue and decides whether to refresh, rewrite, or leave each page.
- **Cost of a wrong call.** A *false positive* wastes an editor's scarce hours on a page that didn't need it. A *false negative* leaves a genuinely declining, high-demand page unattended, quietly bleeding traffic. Because review capacity is tiny next to the candidate pool, ordering errors near the *top* of the list are the expensive ones — which is why the metric that matters is **precision@K**, not overall accuracy.

**One-paragraph frame.** *For a content editor, deciding which pages to review first, we will build a ranked priority score from observable 90-day search and engagement signals, scoring each page's refresh opportunity, measured by precision@K. A wrong call costs wasted editor hours (false positive) or a missed declining page (false negative). A plain fixed rule isn't enough because the signals are many and tangled — demand, position, freshness, CTR and engagement all interact — so a learned ranking can order candidates better than any single if-statement. We will claim only observed / directional / decision-support results.*

In [2]:
# Why this needs RANKING, not a rule that says "fix everything":
# count pages that already look like refresh candidates, then compare that
# pool to what one editor can realistically review in a week.
candidates = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
weekly_capacity = 25  # a generous estimate for one editor

print(f"declining pages with real demand (down & impressions_90d>=100): {len(candidates):,}")
print(f"assumed editor review capacity per week:                        {weekly_capacity}")
print(f"weeks to clear that backlog with NO prioritisation:             {len(candidates)/weekly_capacity:,.0f}")
# The pool dwarfs capacity -> the job is ORDERING, i.e. a ranking/scoring problem.

declining pages with real demand (down & impressions_90d>=100): 13,152
assumed editor review capacity per week:                        25
weeks to clear that backlog with NO prioritisation:             526


## 3. Quick look at the data (2-3 real numbers)

Three numbers, computed live below, that say this lane is worth the next 7 weeks:

1. **The decline signal is common, not rare.** About **54%** of pages carry `trend_direction == "down"`. A problem this widespread is worth prioritising — but it's also *too* widespread to act on blindly, which is exactly what ranking is for. (Note: `trend_direction` is a coarse current-window bucket, so I treat it as a **proxy**, and — per the data dictionary — it can never be a model feature, since the starter label is derived from it.)
2. **Plenty of pages have enough exposure to matter.** About **56%** of pages have `impressions_90d >= 500`, so most of the library is genuinely visible in search — refreshing them protects real traffic, not noise.
3. **Declining pages are the *more* visible ones.** Median 90-day impressions are **961** for declining pages vs **472** for the rest. Decline isn't concentrated in dead pages; it's hitting pages that still pull demand — exactly the pages an editor would regret missing.

Together: a large, visible, actively-declining pool, far bigger than review capacity -> a ranking problem worth solving.

In [3]:
# The three headline numbers, all recomputed straight from the raw CSV.
decline_rate = (df["trend_direction"] == "down").mean()
visible_rate = (df["impressions_90d"] >= 500).mean()
med_declining = df.loc[df["trend_direction"] == "down", "impressions_90d"].median()
med_other = df.loc[df["trend_direction"] != "down", "impressions_90d"].median()

print(f"1. pages declining (trend_direction=='down'): {decline_rate:6.1%}")
print(f"2. pages visible (impressions_90d >= 500):    {visible_rate:6.1%}")
print(f"3. median impressions_90d  declining pages:   {med_declining:,.0f}")
print(f"   median impressions_90d  all other pages:   {med_other:,.0f}")
print()
print("full trend_direction breakdown (proxy signal, NOT a feature):")
print(df["trend_direction"].value_counts().to_string())

1. pages declining (trend_direction=='down'):  54.2%
2. pages visible (impressions_90d >= 500):     55.8%
3. median impressions_90d  declining pages:   961
   median impressions_90d  all other pages:   472

full trend_direction breakdown (proxy signal, NOT a feature):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152


## 4. Careful words: what I can and can't claim

**What this work *can* say:**

- *Observed / measured:* on this 30,000-row anonymized starter slice, a given share of pages are declining, visible, etc. These are counts, not predictions.
- *Directional:* signals like freshness, position and demand *appear to travel with* refresh opportunity. I will say "we observed" and "this suggests", never "this proves".
- *Decision-support:* the output ranks candidates so a human reviews the most promising pages first. It orders a to-do list; it does not automate the edit.

**What this work will *never* claim:**

- **No causal claim.** I cannot say a refresh *caused* a recovery — that needs an experiment or a proper causal design, which this observational data can't provide.
- **Not "predicting Google".** I am scoring refresh opportunity from observable signals, not reverse-engineering or predicting the search algorithm.
- **Scope-bound.** Every number here comes from the small starter slice (32 clients). It is not a benchmark on the full ~79M-row warehouse; any result has to be re-earned there with proper (client-holdout) validation.
- **Proxy honesty.** The starter label (`trend_direction == "down"`) is a coarse *current-window* bucket, not a future outcome. A stronger capstone target would look forward (e.g. features from the prior 90 days -> decline/recovery over the next 30). I'll keep that distinction explicit rather than dressing the proxy up as ground truth.

In [4]:
# A quick honesty check the notebook itself can enforce: the proxy label and
# its parent column must be quarantined from any future feature set.
LABEL_SOURCE = ["trend_direction", "trend_pct"]  # NEVER features (leakage)
ID_COLS = ["content_id", "client_id"]            # grouping/joins only

print("Columns quarantined from features (derive the label -> leakage):")
for c in LABEL_SOURCE:
    print(f"  - {c}")
print("Columns usable for grouping/splits only (pseudonymous ids):")
for c in ID_COLS:
    print(f"  - {c}")
print()
print("Claim posture: observed / directional / decision-support only.")
print("No causal claims. Not predicting Google. Results scoped to the starter slice.")

Columns quarantined from features (derive the label -> leakage):
  - trend_direction
  - trend_pct
Columns usable for grouping/splits only (pseudonymous ids):
  - content_id
  - client_id

Claim posture: observed / directional / decision-support only.
No causal claims. Not predicting Google. Results scoped to the starter slice.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.